## Initial logistic regression experiment

**Goal:** Test whether calendar and pre-storm outage information
can predict a major outage: at least 5% of county customers out
during the episode's 24-hour target window.

**Data:** Train on 2017–2023 and validate on 2024.
The 2025 test labels remain unused in this experiment.

**Inputs:** Month, local start hour, maximum outage share in the
preceding 24 hours, and outage share in the hour before the episode.

**Method:** Replace missing inputs with training-set medians, add
missingness indicators, standardize features, and fit logistic regression.
All preprocessing is fitted on training data only.

**Limitations:** This is an initial retrospective experiment.
Input reporting delays still need verification. It does not yet
test spatial methods.

In [1]:
# =============================================================================
# 05 - BASELINE MODELS: P(county >= 5% out within 24 h | storm episode)
# Summary of results lives in reports/05_baseline_models_summary.md
# Judged on 2024 validation against reports/success_criteria.md (test 2025 is not touched here).
# Every rule / model is logged to Weights & Biases (team project "CAPSTONE MODEL"): metrics, small tables, charts.
# No data or labels are uploaded to W&B. Shared checks live in src/evaluation.py.
# Inputs: processed/model_table/ (from notebook 04). Outputs: reports/baseline_tables/
# =============================================================================

In [1]:
# ---- 0. Setup ---------------------------------------------------------------
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.evaluation import evaluate, log_run, naive_rules  # noqa: E402

TABLES = ROOT / "reports" / "baseline_tables"
TABLES.mkdir(parents=True, exist_ok=True)

LOG_TO_WANDB = False   # False = evaluate locally only (nothing sent to W&B)

# Inputs: the local file when it is in the repo, otherwise streamed from the Hugging Face dataset
HF_DATASET = "hf://datasets/DSCI-Capstone/county-outage-data"


def src(rel):
    """Local path if the file is in the repo, otherwise its hf:// URL."""
    local = ROOT / rel
    return local.as_posix() if local.exists() else f"{HF_DATASET}/{rel}"


TRAIN_PATH = src("processed/model_table/train.parquet")
VAL_PATH = src("processed/model_table/validation.parquet")
train = pd.read_parquet(TRAIN_PATH)
val = pd.read_parquet(VAL_PATH)
DATA_PATHS = [TRAIN_PATH, VAL_PATH]   # hashed into each run's config when local
print(f"train {len(train):,} rows ({train['target'].mean():.1%} positive) | validation {len(val):,} rows ({val['target'].mean():.1%} positive)")

train 208,482 rows (14.1% positive) | validation 31,064 rows (16.5% positive)


In [2]:
# ---- 1. Naive rules (the bar, no ML) -------------------------------------------
# Rates come from training rows only. The family rule uses the highest training rate among the
# storm families present in the episode; the reference rule = family rate x county history / training rate.
rules = naive_rules(train, val)
REF = rules["family_x_history"]

rows = []
for name, p in rules.items():
    m, _ = evaluate(val, p, train)
    rows.append({"rule": name, **{k: m[k] for k in ["pr_auc", "roc_auc", "brier", "brier_skill", "recall_top10", "precision_top10"]}})
naive = pd.DataFrame(rows)
naive.to_csv(TABLES / "naive_rules_validation.csv", index=False)

# First ad-hoc PR-AUC values (before the rules were code); only the reference rule differs, see success_criteria.md
original = {"always_train_rate": 0.165, "family_rate": 0.312, "county_history": 0.368, "family_x_history": 0.472}
naive["original_pr_auc"] = naive["rule"].map(original)
naive.round(4)

,rule,pr_auc,roc_auc,brier,brier_skill,recall_top10,precision_top10,original_pr_auc
0,always_train_rate,0.1648,0.5000,0.1382,-0.0040,0.0928,0.1529,0.165
1,family_rate,0.3115,0.7185,0.1246,0.0950,0.2316,0.3817,0.312
2,county_history,0.3677,0.7472,0.1238,0.1003,0.2693,0.4438,0.368
3,family_x_history,0.4691,0.8041,0.1116,0.1896,0.3258,0.5369,0.472


In [4]:
# ---- 2. Log the naive rules to W&B ----------------------------------------------
RULE_NOTES = {
    "always_train_rate": "constant = training positive rate",
    "family_rate": "max training outage rate among storm families present in the episode",
    "county_history": "hist_outage_rate (county past outage rate, shrunk to national; leave-one-year-out in train)",
    "family_x_history": "family_rate * county_history / training rate, clipped to [0, 1] (reference rule)",
}
if LOG_TO_WANDB:
    for name, p in rules.items():
        config = {"model": name, "model_type": "naive_rule", "rule": RULE_NOTES[name],
                  "train_rows": len(train), "train_pos_rate": train["target"].mean()}
        _, _, url = log_run(f"naive_{name}", "baseline", val, p, train, config,
                            p_ref=None if name == "family_x_history" else REF, data_paths=DATA_PATHS)
        print(name, "->", url)

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\chris\_netrc.


wandb: Currently logged in as: chriswalker5705 to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


wandb: Tracking run with wandb version 0.31.0


wandb: Run data is saved locally in D:\Capstone-Power-Outage-Prediction\notebooks\wandb\run-20261025_231450-rurqa546
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run naive_always_train_rate


...

wandb: uploading artifact run-5nl32hop-by_year


wandb: 
wandb: Run summary:
wandb:                     brier 0.11156
wandb:               brier_skill 0.18957
wandb:           cal_overall_pts -0.33397
wandb:      cal_worst_decile_pts 3.51233
wandb:          max_recall_top10 0.60684
wandb:            pass_1_ranking False
wandb:        pass_2_brier_skill False
wandb:       pass_2b_calibration True
wandb: pass_3_no_hidden_failures True
wandb:       pass_5_recall_top10 False
wandb:                        +7 ...
wandb: 


wandb:  View run naive_family_x_history at: https://forge.coreweave.com/wandb/leslie-ortega-us-georgia-state-university/CAPSTONE%20MODEL/runs/5nl32hop
wandb:  View project at: https://forge.coreweave.com/wandb/leslie-ortega-us-georgia-state-university/CAPSTONE%20MODEL
wandb: Synced 4 W&B file(s), 6 media file(s), 8 artifact file(s) and 0 other file(s)


wandb: Find logs at: .\wandb\run-20261025_231514-5nl32hop\logs


family_x_history -> https://forge.coreweave.com/wandb/leslie-ortega-us-georgia-state-university/CAPSTONE%20MODEL/runs/5nl32hop


In [3]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

features = ["month", "local_hour", "pre_peak_24h", "pct_out_at_start"]

model = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True),
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

model.fit(train[features], train["target"])

print("Logistic regression trained.")

Logistic regression trained.


In [4]:
predictions = model.predict_proba(val[features])[:, 1]

metrics, _ = evaluate(val, predictions, train)

comparison = {
    "rule": "logistic_regression",
    **{
        key: metrics[key]
        for key in [
            "pr_auc", "roc_auc", "brier",
            "brier_skill", "recall_top10", "precision_top10"
        ]
    }
}

pd.concat(
    [naive.drop(columns="original_pr_auc"), pd.DataFrame([comparison])],
    ignore_index=True
).round(4)

,rule,pr_auc,roc_auc,brier,brier_skill,recall_top10,precision_top10
0,always_train_rate,0.1648,0.5000,0.1382,-0.0040,0.0928,0.1529
1,family_rate,0.3115,0.7185,0.1246,0.0950,0.2316,0.3817
2,county_history,0.3677,0.7472,0.1238,0.1003,0.2693,0.4438
3,family_x_history,0.4691,0.8041,0.1116,0.1896,0.3258,0.5369
4,logistic_regression,0.3636,0.6600,0.1254,0.0889,0.2424,0.3994


In [5]:
results = pd.DataFrame([comparison])
results["features"] = ", ".join(features)
results["train_years"] = "2017–2023"
results["validation_year"] = 2024

results.to_csv(
    TABLES / "logistic_regression_initial_validation.csv",
    index=False
)

print("First model results saved.")

First model results saved.


In [6]:
from sklearn.metrics import confusion_matrix

# Initial diagnostic threshold, not our final alert threshold
predicted_labels = predictions >= 0.5

print(pd.DataFrame(
    confusion_matrix(val["target"], predicted_labels, labels=[0, 1]),
    index=["Actual: no major outage", "Actual: major outage"],
    columns=["Predicted: no major outage", "Predicted: major outage"]
))

                         Predicted: no major outage  Predicted: major outage
Actual: no major outage                       25839                      105
Actual: major outage                           4595                      525


In [7]:
from sklearn.metrics import confusion_matrix

rows = []

for threshold in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    alerts = predictions >= threshold
    tn, fp, fn, tp = confusion_matrix(
        val["target"], alerts, labels=[0, 1]
    ).ravel()

    rows.append({
        "threshold": threshold,
        "recall": tp / (tp + fn),
        "precision": tp / (tp + fp) if tp + fp else float("nan"),
        "missed_outages": fn,
        "false_alarms": fp,
        "false_alarms_per_1000": 1000 * fp / len(val)
    })

pd.DataFrame(rows).round(3)

,threshold,recall,precision,missed_outages,false_alarms,false_alarms_per_1000
0,0.05,1.000,0.165,0,25944,835.179
1,0.10,0.930,0.180,356,21634,696.433
2,0.15,0.338,0.323,3388,3634,116.984
3,0.20,0.183,0.563,4183,728,23.435
4,0.30,0.134,0.682,4433,320,10.301
5,0.50,0.103,0.833,4595,105,3.380


## Results and interpretation

On 2024 validation data, the initial logistic regression achieved
PR-AUC of 0.3636 and Brier score of 0.1254.

It improved on the constant-probability baseline but underperformed
the strongest family × county-history rule (PR-AUC 0.4691;
Brier score 0.1116). These approaches use different information,
so this is not a controlled comparison of algorithms alone.

At an alert threshold of 0.50, recall was 10.3% and precision was
83.3%. Lowering the threshold to 0.10 increased recall to 93.0%
but produced 21,634 false alarms.

No final alert threshold was selected. Threshold adjustment alone
did not provide a satisfactory balance for this initial model.

Next steps:
- Verify feature availability and reporting-delay assumptions.
- Compare improved models using consistent inputs and evaluation rows.
- Test spatial features or methods.
- Coordinate experiment tracking and runtime measurements with the team.
- Keep 2025 test labels reserved for final evaluation.

In [8]:
pd.DataFrame(rows).to_csv(
    TABLES / "logistic_regression_threshold_comparison.csv",
    index=False
)

In [ ]:
# ---- 3. Log the logistic regression to W&B ----------------------------------------
# Same checks as the naive rules; the bootstrap CI compares against the reference rule (family_x_history).
if LOG_TO_WANDB:
    lr = model.named_steps["logisticregression"]
    config = {"model": "logistic_regression", "model_type": "logistic_regression",
              "features": features, "n_features": len(features),
              "preprocessing": "median impute + missing indicators, standard scaling (fit on train)",
              "C": lr.C, "penalty": lr.penalty, "solver": lr.solver, "max_iter": lr.max_iter,
              "class_weight": str(lr.class_weight),
              "train_rows": len(train), "train_pos_rate": train["target"].mean()}
    _, _, url = log_run("logistic_regression_initial", "baseline", val, predictions, train, config,
                        p_ref=REF, data_paths=DATA_PATHS,
                        extra_tables={"threshold_comparison": pd.DataFrame(rows)})
    print("logistic_regression ->", url)